# Ficheros · Agenda, plantillas e intérprete

Persistencia de datos entre ejecuciones, cartas personalizadas, procesamiento de HTML y un intérprete para un lenguaje sencillo.

## 1. Cartas personalizadas

`clientes.txt` tiene en cada línea el correo, el sexo (`H`/`M`) y el nombre de un cliente; `carta.txt` contiene `$CLIENTE$` donde va el nombre y las formas `don/doña`, `Estimado/a` y `Sr/a`. Generar cada carta personalizada según el sexo. En lugar de enviar los correos de verdad, se guardan en ficheros `.eml` listos para enviar (con `smtplib` se podrían mandar).

In [1]:
from email.message import EmailMessage

with open("clientes.txt", "w", encoding="utf-8") as f:
    f.write("al00000@example.com H Pedro Pérez\ncliente@example.com M Ana Gil\n")
with open("carta.txt", "w", encoding="utf-8") as f:
    f.write("Estimado/a Sr/a $CLIENTE$:\n\nTenemos noticias de que ud., don/doña $CLIENTE$, no ha abonado la cuota mensual.\n")

def personaliza(carta, nombre, sexo):
    hombre = sexo == "H"
    for generico, h, m in (("don/doña", "don", "doña"), ("Estimado/a", "Estimado", "Estimada"), ("Sr/a", "Sr", "Sra")):
        carta = carta.replace(generico, h if hombre else m)
    return carta.replace("$CLIENTE$", nombre)

def genera_correos(clientes, plantilla):
    with open(plantilla, encoding="utf-8") as f:
        carta = f.read()
    with open(clientes, encoding="utf-8") as f:
        for linea in f:
            correo, sexo, nombre = linea.rstrip("\n").split(" ", 2)
            mensaje = EmailMessage()
            mensaje["To"], mensaje["Subject"] = correo, "Aviso de pago"
            mensaje.set_content(personaliza(carta, nombre, sexo))
            with open(f"{correo}.eml", "wb") as g:
                g.write(bytes(mensaje))
            print(personaliza(carta, nombre, sexo))

genera_correos("clientes.txt", "carta.txt")

Estimado Sr Pedro Pérez:

Tenemos noticias de que ud., don Pedro Pérez, no ha abonado la cuota mensual.

Estimada Sra Ana Gil:

Tenemos noticias de que ud., doña Ana Gil, no ha abonado la cuota mensual.



## 2. Agenda en fichero

Agenda guardada en `agenda.txt` con una línea por entrada: `nombre:apellido:teléfono1:teléfono2...` (formato similar a `/etc/passwd`). Operaciones: añadir (si la persona ya existe, se sustituyen sus teléfonos), buscar (devuelve la lista de teléfonos), borrar un teléfono, listar ordenado por apellido (una línea por entrada) y listar los teléfonos de quienes tienen el apellido empezando por una letra. La agenda se mantiene ordenada al añadir.

In [2]:
AGENDA = "agenda.txt"

def carga():
    try:
        with open(AGENDA, encoding="utf-8") as f:
            return [linea.rstrip("\n").split(":") for linea in f if linea.strip()]
    except FileNotFoundError:
        return []

def guarda(entradas):
    entradas.sort(key=lambda e: (e[1], e[0]))
    with open(AGENDA, "w", encoding="utf-8") as f:
        f.writelines(":".join(e) + "\n" for e in entradas)

def anyadir_entrada(nombre, apellido, telefonos):
    entradas = [e for e in carga() if e[:2] != [nombre, apellido]]
    entradas.append([nombre, apellido, *telefonos])
    guarda(entradas)

def buscar_entrada(nombre, apellido):
    return next((e[2:] for e in carga() if e[:2] == [nombre, apellido]), [])

def borrar_telefono(nombre, apellido, telefono):
    entradas = carga()
    for e in entradas:
        if e[:2] == [nombre, apellido] and telefono in e[2:]:
            e.remove(telefono)
    guarda(entradas)

def listado():
    for nombre, apellido, *telefonos in carga():
        print(f"{apellido}, {nombre}: {' '.join(telefonos)}")

def telefonos_por_letra(letra):
    return {f"{e[1]}, {e[0]}": e[2:] for e in carga() if e[1].upper().startswith(letra.upper())}

anyadir_entrada("Pedro", "López", ["964112537", "964009923", "96411092"])
anyadir_entrada("Ana", "Gil", ["600111222"])
anyadir_entrada("Ana", "Gil", ["600333444"])
anyadir_entrada("Luis", "Lara", ["700000000"])
borrar_telefono("Pedro", "López", "964009923")
listado()
print(buscar_entrada("Pedro", "López"), telefonos_por_letra("l"))

Gil, Ana: 600333444
Lara, Luis: 700000000
López, Pedro: 964112537 96411092
['964112537', '96411092'] {'Lara, Luis': ['700000000'], 'López, Pedro': ['964112537', '96411092']}


## 3. Agenda en HTML

Generar `agenda.html` con la agenda ordenada por apellido, una sección por letra inicial y el apellido en negrita.

In [3]:
from itertools import groupby

def agenda_html(ruta="agenda.html"):
    partes = ["<html><body><h1>Agenda</h1>"]
    for letra, grupo in groupby(sorted(carga(), key=lambda e: e[1]), key=lambda e: e[1][0].upper()):
        partes.append(f"<h2>{letra}</h2>")
        partes += [f"<p><b>{apellido}</b>, {nombre}: {' '.join(tels)}</p>" for nombre, apellido, *tels in grupo]
    partes.append("</body></html>")
    with open(ruta, "w", encoding="utf-8") as f:
        f.write("\n".join(partes))
    return ruta

print(open(agenda_html(), encoding="utf-8").read())

<html><body><h1>Agenda</h1>
<h2>G</h2>
<p><b>Gil</b>, Ana: 600333444</p>
<h2>L</h2>
<p><b>Lara</b>, Luis: 700000000</p>
<p><b>López</b>, Pedro: 964112537 96411092</p>
</body></html>


## 4. Copias con `write`

¿Funciona igual escribir tres líneas con tres llamadas a `write` que con una sola `write(linea1 + linea2 + linea3)`? Sí: `write` no añade nada entre llamadas.

In [4]:
lineas = ["Pedro\n", "López\n", "964112537\n"]
with open("a.txt", "w", encoding="utf-8") as f:
    for l in lineas:
        f.write(l)
with open("b.txt", "w", encoding="utf-8") as f:
    f.write(lineas[0] + lineas[1] + lineas[2])
print(open("a.txt", encoding="utf-8").read() == open("b.txt", encoding="utf-8").read())

True


## 5. Recordar datos entre ejecuciones

Hacer que la gestión de estudiantes y la del videoclub conserven sus datos entre ejecuciones guardándolos en ficheros (uno para películas y otro para socios). JSON es un formato de texto legible y estándar.

In [5]:
import json

def guarda_json(ruta, datos):
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=1)

def carga_json(ruta, por_defecto):
    try:
        with open(ruta, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return por_defecto

estudiantes = carga_json("estudiantes.json", [])
estudiantes.append({"nombre": "Ana", "grupo": "A", "examen": 8.5, "practica": True})
guarda_json("estudiantes.json", estudiantes)
guarda_json("peliculas.json", [{"titulo": "Matrix", "generos": ["ciencia ficción"], "ejemplares": 2}])
guarda_json("socios.json", [{"dni": "1Z", "nombre": "Ana"}])
print(carga_json("estudiantes.json", []), carga_json("peliculas.json", []), carga_json("socios.json", []))

[{'nombre': 'Ana', 'grupo': 'A', 'examen': 8.5, 'practica': True}] [{'titulo': 'Matrix', 'generos': ['ciencia ficción'], 'ejemplares': 2}] [{'dni': '1Z', 'nombre': 'Ana'}]


## 6. Procesar HTML

En un fichero HTML: cambiar las negritas (`<b>`) por cursivas (`<i>`); mostrar el texto de las cabeceras de nivel 1; y mostrar todas las cabeceras sangradas dos espacios por nivel.

In [6]:
import re

html = """<HTML>
<BODY>
<H1>Un titular</H1>
<P>Texto en un <b>párrafo</b>.
<P>Otro <B>párrafo</B>.
<H1>Otro titular</H1>
<H2>Un subtítulo</H2>
<P>Y su texto.
<H3>Un subsubtítulo</H3>
<H2>Otro subtítulo</H2>
<P>Y el suyo
</BODY>
</HTML>"""
with open("pagina.html", "w", encoding="utf-8") as f:
    f.write(html)

with open("pagina.html", encoding="utf-8") as f:
    contenido = f.read()
with open("pagina_cursiva.html", "w", encoding="utf-8") as f:
    f.write(re.sub(r"<(/?)b>", r"<\1i>", contenido, flags=re.IGNORECASE))
print([l for l in open("pagina_cursiva.html", encoding="utf-8") if "<i>" in l])

cabeceras = re.findall(r"<H([1-6])>(.*?)</H\1>", contenido, flags=re.IGNORECASE)
print([texto for nivel, texto in cabeceras if nivel == "1"])
for nivel, texto in cabeceras:
    print("  " * (int(nivel) - 1) + texto)

['<P>Texto en un <i>párrafo</i>.\n', '<P>Otro <i>párrafo</i>.\n']
['Un titular', 'Otro titular']
Un titular
Otro titular
  Un subtítulo
    Un subsubtítulo
  Otro subtítulo


## 7. Intérprete de un lenguaje sencillo

Interpretar programas con tres variables (A, B y C) y las sentencias `asigna V op x y y` (con `suma`, `resta`, `producto` y `division` entera), `muestra V` y `si valor condicion valor entonces linea n` (con `igual` o `distinto`), donde los valores pueden ser números o variables.

In [7]:
def interpreta(programa):
    variables = {"A": 0, "B": 0, "C": 0}
    operaciones = {"suma": lambda a, b: a + b, "resta": lambda a, b: a - b,
                   "producto": lambda a, b: a * b, "division": lambda a, b: a // b}
    valor = lambda t: variables[t] if t in variables else int(t)
    lineas = [l.split() for l in programa.strip().splitlines()]
    pc = 0
    while pc < len(lineas):
        palabras, pc = lineas[pc], pc + 1
        if palabras[0] == "asigna":
            _, var, op, x, _, y = palabras
            variables[var] = operaciones[op](valor(x), valor(y))
        elif palabras[0] == "muestra":
            print(variables[palabras[1]])
        elif palabras[0] == "si":
            _, x, cond, y, _, _, numero = palabras
            if (valor(x) == valor(y)) == (cond == "igual"):
                # las líneas se numeran desde 1
                pc = int(numero) - 1
    return variables

print(interpreta("""
asigna A suma 3 y 7
asigna B resta A y 2
asigna C producto A y B
asigna A division A y 10
muestra A
muestra B
"""))
interpreta("""
asigna A suma 0 y 1
asigna B suma 0 y 1
muestra B
asigna B producto 2 y B
asigna A suma A y 1
si A distinto 8 entonces linea 3
""")

1
8
{'A': 1, 'B': 8, 'C': 80}
1
2
4
8
16
32
64
